# preborn_train — GPU fork of the Prod pipeline_train (one core per role)

Runs on an ML/GPU cluster (torch cu128 + mostlyai[local]==6.1.1). Trains ONE
DP core generator for the role's cfg (mother or infant) — PREBORN has NO
reference group, so there is a single `core.zip` per role. Writes `gen_*` into
the role's working schema and records the generation in that role's manifest.

Sequence (per role): run `preborn_pipeline role=<r> stage=pre` (serverless) →
this notebook `role=<r>` (GPU) → `preborn_pipeline role=<r> stage=post`.

DP: `differential_privacy` is injected by the engine's `build_table_config` on
every core table (asserted below). Composed epsilon is reported per generator;
the two generators are independent DP budgets. No DP mechanism runs outside this
training step.

In [0]:
%pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128
%pip install 'mostlyai[local]==6.1.1'
%restart_python

In [0]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
dbutils.widgets.dropdown("role", "mother", ["mother", "infant"])
dbutils.widgets.dropdown("allow_cpu", "false", ["false", "true"])
dbutils.widgets.text("max_training_time", "120")
dbutils.widgets.text("max_epochs", "50")
dbutils.widgets.text("local_dir", "/local_disk0/mostlyai")

import torch
gpu_ok = torch.cuda.is_available()
print(f"torch {torch.__version__} | cuda available: {gpu_ok}")
if gpu_ok:
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| VRAM GB:", round(p.total_memory / 1e9, 1))
elif dbutils.widgets.get("allow_cpu") != "true":
    raise RuntimeError("torch.cuda.is_available() is False — set allow_cpu=true to override (slow)")

In [0]:
%run /Workspace/Shared/ADC-DB/Prod/Synthetic/engine

In [0]:
%run ./preborn_config

In [0]:
import json
from datetime import datetime, timezone
from mostlyai.sdk import MostlyAI

role = dbutils.widgets.get("role")
cfg = PREBORN_MOTHER_CONFIG if role == "mother" else PREBORN_INFANT_CONFIG
# engine reads model_limits from cfg (widget overrides)
cfg.model_limits["max_training_time"] = int(dbutils.widgets.get("max_training_time"))
cfg.model_limits["max_epochs"] = int(dbutils.widgets.get("max_epochs"))
LOCAL_DIR = dbutils.widgets.get("local_dir")

TGT = f"{cfg.target['catalog']}.{cfg.target['schema']}"
VOL = cfg.target["volume"]
GEN_DIR = f"{VOL}/generators"
GEN_CORE = f"{GEN_DIR}/core.zip"
CONFIG_DIR = f"{VOL}/config"
REPORT_DIR = f"{VOL}/reports"
OUTPUT_PERSONS = cfg.extra["output"]["persons"]
assert cfg.dp["enabled"] and cfg.dp["delta"] == 1e-6, "DP config drift — STOP"
for d in (GEN_DIR, CONFIG_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

def _utcnow():
    return datetime.now(timezone.utc).isoformat()

def read_manifest():
    try:
        return json.load(open(f"{CONFIG_DIR}/run_manifest.json"))
    except Exception:
        return None

def write_manifest(m):
    with open(f"{CONFIG_DIR}/run_manifest.json", "w") as f:
        json.dump(m, f, indent=2, default=str)

print(f"role={role} TGT={TGT} output_persons={OUTPUT_PERSONS}")

In [0]:
def train_core(mostly, sdtypes):
    frames = prepare_frames("core")
    total = sum(len(f) for f in frames.values())
    assert total <= cfg.extra["max_pandas_rows"], f"cohort too large ({total})"
    tables = build_generator_tables("core", sdtypes, frames)
    # DP boundary: every core table must carry differential_privacy
    dp_tables = [t["name"] for t in tables
                 if "differential_privacy" in t.get("tabular_model_configuration", {})]
    assert len(dp_tables) == len(tables), \
        f"DP must be on EVERY core table: {len(dp_tables)}/{len(tables)}"
    ce = composed_epsilon([t["name"] for t in tables])
    print(f"[{role}] composed epsilon (naive sum incl profiler): {ce}")
    g = mostly.train(config={"name": f"{cfg.name}_core", "tables": tables}, start=False)
    g.training.start(); g.training.wait(progress_bar=True)
    status = _val(getattr(mostly.generators.get(g.id), "training_status", None))
    if status != "DONE":
        raise RuntimeError(f"training status={status} (not DONE) — refusing to export")
    g.export_to_file(GEN_CORE)
    g.reports(file_path=f"{REPORT_DIR}/core_qa", display=False)
    with open(f"{REPORT_DIR}/dp_audit.json", "w") as f:
        json.dump({"enabled": True, "n_dp_tables": len(dp_tables), "n_core_tables": len(tables),
                   "composed_epsilon_naive_sum": ce, "role": role}, f, indent=2, default=str)
    return {"composed_epsilon": ce, "training_status": status}

def _val(x):
    return getattr(x, "value", x)

def generate(mostly):
    assert os.path.exists(GEN_CORE), f"{GEN_CORE} missing — train first"
    gcore = mostly.generators.import_from_file(GEN_CORE)
    sd = mostly.generate(gcore, size={cfg.subject: OUTPUT_PERSONS})
    data = sd.data()
    counts = {}
    for name in topo_order(cfg):
        pdf = data[name] if isinstance(data, dict) else sd.data(name)
        (spark.createDataFrame(pdf).write.mode("overwrite")
         .option("overwriteSchema", "true").saveAsTable(f"{TGT}.gen_{name}"))
        counts[name] = int(len(pdf))
        print(f"  gen_{name}: {len(pdf)}")
    empty = [t for t, n in counts.items() if n == 0]
    assert not empty, f"empty generated tables: {empty}"
    return counts

In [0]:
mostly = MostlyAI(local=True, local_dir=LOCAL_DIR)
sdtypes = load_sdtypes()
train_rec = train_core(mostly, sdtypes)
counts = generate(mostly)

m = read_manifest() or {"run_id": "train_" + role}
m["train"] = {**train_rec, "recorded_utc": _utcnow()}
m["generation"] = {"recorded_utc": _utcnow(), "kind": "mostlyai",
                   "output_persons": OUTPUT_PERSONS, "tables": counts}
m["stage"] = "generated"
write_manifest(m)
print(f"[{role}] TRAIN+GENERATE complete — gen_* written, manifest updated")
dbutils.notebook.exit(json.dumps({"role": role, "counts": counts, "train": train_rec}, default=str))